<a href="https://sigmoidal.ai">
  <img src="https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/_assets/logo_sigmoidal.png" alt="Sigmoidal" width="300">
</a>

# CT Scan com Python: Como Visualizar Tomografias 3D

**Autor:** Carlos Melo — [sigmoidal.ai](https://sigmoidal.ai)

In [ ]:
!pip install SimpleITK -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk

In [ ]:
import gdown
import os

os.makedirs("data", exist_ok=True)

# MHD (header)
gdown.download(
    "https://drive.google.com/uc?id=1KIuaToQzioFQbQg6b_DiTLzN_2_0y7ph",
    "data/1.3.6.1.4.1.14519.5.2.1.6279.6001.108197895896446896160048741492.mhd",
    quiet=False,
)

# RAW (voxels)
gdown.download(
    "https://drive.google.com/uc?id=1Iau3f1Byl0HMluwD5RRhCEalnt2ky4xI",
    "data/1.3.6.1.4.1.14519.5.2.1.6279.6001.108197895896446896160048741492.raw",
    quiet=False,
)

print("Download concluido!")

In [ ]:
mhd_path = "/content/data/1.3.6.1.4.1.14519.5.2.1.6279.6001.108197895896446896160048741492.mhd"
ct_sitk = sitk.ReadImage(mhd_path)
ct_array = sitk.GetArrayFromImage(ct_sitk)


In [ ]:
ct_array.shape

In [ ]:
ct_array.dtype

In [ ]:
print(f"{ct_array.ndim}D")

In [ ]:
spacing = ct_sitk.GetSpacing()
origin = ct_sitk.GetOrigin()

print(f"Espaçamento (x, y, z): {spacing} mm ")
print(f"Origem: {origin}")


In [ ]:
fatia = ct_array[0]
print(f"Min: {fatia.min()}, Max: {fatia.max()}")

In [ ]:
plt.imshow(fatia, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
n_fatias = ct_array.shape[0]
indices = [0, n_fatias // 4, n_fatias // 2, 3 * n_fatias // 4, n_fatias - 1]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, idx in zip(axes, indices):
    ax.imshow(ct_array[idx], cmap="gray")
    ax.set_title(f"Fatia {idx}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
print(f"MIN: {ct_array.min()}")
print(f"MAX: {ct_array.max()}")

In [ ]:
plt.hist(ct_array.flatten(), bins=100);

In [ ]:
def aplicar_janela(img, center, width):
    lower = center - width // 2
    upper = center + width // 2
    img_janelada = np.clip(img, lower, upper)
    return img_janelada

In [ ]:
fatia_meio = ct_array[ct_array.shape[0] // 2]

janelas = {
    "Pulmao (C:-600, W:1500)": (-600, 1500),
    "Mediastino (C:40, W:400)": (40, 400),
    "Osso (C:400, W:1800)": (400, 1800),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, (nome, (center, width)) in zip(axes, janelas.items()):
    img = aplicar_janela(fatia_meio, center, width)
    ax.imshow(img, cmap="gray")
    ax.set_title(nome)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
d, h, w = ct_array.shape

axial = ct_array[d // 2]         # fatia no meio do eixo z
sagital = ct_array[:, :, w // 2]  # fatia no meio do eixo x
coronal = ct_array[:, h // 2, :]  # fatia no meio do eixo y

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(axial, cmap="gray")
axes[0].set_title("Axial")
axes[1].imshow(sagital, cmap="gray")
axes[1].set_title("Sagital")
axes[2].imshow(coronal, cmap="gray")
axes[2].set_title("Coronal")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()